In [ ]:
# APF3 - Modelo predictivo de morosidad para la Cooperativa A.C.S.
# Curso: Innovacion y Transformacion Digital
# Autor: Equipo del proyecto
# Este script es reproducible: genera el dataset simulado, entrena modelos y exporta metricas/graficos.


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1. Construccion de dataset simulado
# La cooperativa actualmente trabaja con cuadernos fisicos y Excel, por eso se simulan datos
# coherentes con el proceso de cobranza: pagos puntuales, atrasos, saldo pendiente y canal de pago.
n = 320
socios = np.arange(1, n + 1)
sectores = np.random.choice(['A', 'B', 'C', 'D'], n, p=[0.25, 0.32, 0.28, 0.15])
tipo_puesto = np.random.choice(['alimentos', 'ropa', 'servicios', 'otros'], n, p=[0.35, 0.30, 0.20, 0.15])
antiguedad = np.random.randint(6, 121, n)
monto_cuota = np.random.choice([50, 60, 70, 80, 100], n, p=[0.35, 0.25, 0.20, 0.15, 0.05])
pagos_puntuales_6m = np.clip(np.random.poisson(4.2, n), 0, 6)
aportes_atrasados_6m = np.clip(
    6 - pagos_puntuales_6m + np.random.binomial(1, 0.15, n) - np.random.binomial(1, 0.10, n),
    0,
    6,
)
dias_retraso_promedio = np.clip(np.random.normal(5 + aportes_atrasados_6m * 5, 5, n), 0, 45).round(1)
saldo_pendiente = (aportes_atrasados_6m * monto_cuota + np.maximum(0, np.random.normal(0, 20, n))).round(2)
recibio_recordatorio = np.random.choice([0, 1], n, p=[0.38, 0.62])
canal_pago = np.random.choice(['presencial', 'transferencia', 'yape_plin'], n, p=[0.65, 0.20, 0.15])

# Variable objetivo: moroso = 1 si el socio presenta riesgo de caer en mora; 0 si no.
# La probabilidad depende de variables razonables del negocio.
logit = (
    -2.2
    + 0.09 * dias_retraso_promedio
    + 0.45 * aportes_atrasados_6m
    + 0.003 * saldo_pendiente
    - 0.45 * recibio_recordatorio
    - 0.35 * (canal_pago != 'presencial')
    - 0.12 * pagos_puntuales_6m
)
prob_mora = 1 / (1 + np.exp(-logit))
moroso = np.random.binomial(1, prob_mora)

df = pd.DataFrame({
    'id_socio': socios,
    'sector': sectores,
    'tipo_puesto': tipo_puesto,
    'antiguedad_meses': antiguedad,
    'monto_cuota_mensual': monto_cuota,
    'pagos_puntuales_6m': pagos_puntuales_6m,
    'aportes_atrasados_6m': aportes_atrasados_6m,
    'dias_retraso_promedio': dias_retraso_promedio,
    'saldo_pendiente': saldo_pendiente,
    'recibio_recordatorio': recibio_recordatorio,
    'canal_pago': canal_pago,
    'moroso': moroso,
})

# Ingenieria de caracteristicas
# Estas variables resumen el comportamiento de pago y son faciles de entender por la cooperativa.
df['indice_historial_pago'] = (
    df['pagos_puntuales_6m'] / (df['pagos_puntuales_6m'] + df['aportes_atrasados_6m'] + 1)
).round(3)
df['ratio_saldo_cuota'] = (df['saldo_pendiente'] / df['monto_cuota_mensual']).round(2)

# AQUI ESTA EL PRIMER CAMBIO: sep=';'
df.to_csv('dataset_morosidad_cooperativa_acs.csv', index=False, sep=';')

# 2. Analisis exploratorio basico
print('Dimensiones del dataset:', df.shape)
print('\nDistribucion del target moroso:')
print(df['moroso'].value_counts())
print('\nEstadisticos descriptivos:')
print(df.select_dtypes(include='number').describe().round(2))

# Grafico 1: distribucion del target
plt.figure(figsize=(6, 4))
df['moroso'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribucion de la variable objetivo: moroso')
plt.xlabel('Moroso (0 = No, 1 = Si)')
plt.ylabel('Numero de socios')
plt.tight_layout()
plt.savefig('grafico_distribucion_target.png', dpi=160)
plt.close()

# Grafico 2: correlacion de variables numericas
corr = df.select_dtypes(include='number').drop(columns=['id_socio']).corr()
plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Correlacion')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Mapa de correlacion de variables numericas')
plt.tight_layout()
plt.savefig('grafico_correlacion.png', dpi=160)
plt.close()

# 3. Preprocesamiento y particion
X = df.drop(columns=['moroso', 'id_socio'])
y = df['moroso']
cat_cols = ['sector', 'tipo_puesto', 'canal_pago']
num_cols = [col for col in X.columns if col not in cat_cols]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

# 4. Modelos comparables de la Unidad 3
models = {
    'Regresion Logistica': (
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        {'model__C': [0.1, 1, 10], 'model__class_weight': [None, 'balanced']},
    ),
    'SVM': (
        SVC(random_state=RANDOM_STATE, probability=True),
        {'model__C': [0.5, 1, 2], 'model__kernel': ['rbf', 'linear'], 'model__class_weight': [None, 'balanced']},
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [3, 5, 7], 'model__weights': ['uniform', 'distance']},
    ),
    'Arbol de Decision': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {'model__max_depth': [3, 4, 5, None], 'model__min_samples_leaf': [1, 5, 10], 'model__class_weight': [None, 'balanced']},
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'model__n_estimators': [50, 100], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]},
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
best_estimators = {}

for name, (model, params) in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    search = GridSearchCV(pipe, params, scoring='f1', cv=cv, n_jobs=1)
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)

    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'Mejores_parametros': search.best_params_,
    })
    best_estimators[name] = search.best_estimator_

metrics_df = pd.DataFrame(results).sort_values('F1', ascending=False)

# AQUI ESTA EL SEGUNDO CAMBIO: sep=';'
metrics_df.to_csv('metricas_modelos_morosidad.csv', index=False, sep=';')

print('\nMetricas por modelo:')
print(metrics_df.round(3))

best_model_name = metrics_df.iloc[0]['Modelo']
best_model = best_estimators[best_model_name]
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
print('\nModelo seleccionado:', best_model_name)
print('Matriz de confusion:')
print(cm)

# Grafico 3: matriz de confusion
ConfusionMatrixDisplay(cm, display_labels=['No moroso', 'Moroso']).plot(values_format='d')
plt.title(f'Matriz de confusion - {best_model_name}')
plt.tight_layout()
plt.savefig('grafico_matriz_confusion.png', dpi=160)
plt.close()

# Grafico 4: comparacion F1 por modelo
plt.figure(figsize=(7, 4))
metrics_df.set_index('Modelo')['F1'].plot(kind='bar')
plt.title('Comparacion de F1 por modelo')
plt.xlabel('Modelo')
plt.ylabel('F1')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('grafico_comparacion_f1.png', dpi=160)
plt.close()

joblib.dump(best_model, 'modelo_morosidad_acs.joblib')
print('\nArchivos generados: dataset, metricas, graficos y modelo joblib.')

Dimensiones del dataset: (320, 14)

Distribucion del target moroso:
moroso
0    189
1    131
Name: count, dtype: int64

Estadisticos descriptivos:
       id_socio  antiguedad_meses  monto_cuota_mensual  pagos_puntuales_6m  \
count    320.00            320.00               320.00              320.00   
mean     160.50             61.10                64.44                3.99   
std       92.52             33.98                13.97                1.58   
min        1.00              6.00                50.00                0.00   
25%       80.75             31.75                50.00                3.00   
50%      160.50             60.00                60.00                4.00   
75%      240.25             91.00                70.00                5.00   
max      320.00            120.00               100.00                6.00   

       aportes_atrasados_6m  dias_retraso_promedio  saldo_pendiente  \
count                320.00                 320.00           320.00   
mean    

C:\Users\RYZEN\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\RYZEN\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\RYZEN\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\RYZEN\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probabilit


Metricas por modelo:
                Modelo  Accuracy  Precision  Recall     F1  \
3    Arbol de Decision     0.825      0.788   0.788  0.788   
4    Gradient Boosting     0.800      0.815   0.667  0.733   
1                  SVM     0.788      0.767   0.697  0.730   
0  Regresion Logistica     0.775      0.742   0.697  0.719   
2                  KNN     0.725      0.704   0.576  0.633   

                                  Mejores_parametros  
3  {'model__class_weight': None, 'model__max_dept...  
4  {'model__learning_rate': 0.05, 'model__max_dep...  
1  {'model__C': 0.5, 'model__class_weight': None,...  
0      {'model__C': 10, 'model__class_weight': None}  
2  {'model__n_neighbors': 3, 'model__weights': 'd...  

Modelo seleccionado: Arbol de Decision
Matriz de confusion:
[[40  7]
 [ 7 26]]

Archivos generados: dataset, metricas, graficos y modelo joblib.
